# SALM Fine-tuning 09 · Name that genre: when fine-tuning makes it worse, probe instead (GTZAN)

> **Task** 30-s music clip → one of 10 genres · **Data** GTZAN, fault-filtered (artist-disjoint) split
> **Metric** accuracy ↑ · **Result** 78.97% → **86.21%** (+9.2%) · **Time** ~1 h, no training

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: a model that already knows more than we can teach it

GTZAN is the classic music-genre benchmark: *blues, classical, country, disco, hip-hop, jazz, metal, pop, reggae,
rock*, as thirty-second clips. We use the **fault-filtered split**, which removes duplicates and mislabels and keeps
artists out of more than one split. It's small: 443 training clips.

The prompted base model is surprisingly good: **79.0%** on test (with the prompt chosen on dev), already close to the
~79-80% that strong published systems reach on this split. With so little data and such a strong prior, weight
fine-tuning has little room to help and can do harm. This tutorial shows how to recognize that situation and what to
do instead: keep the model frozen and fit a small linear probe on its label probabilities.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data

In [ ]:
d = M / "gtzan"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py gtzan", log="prep_gtzan.log")
P2 = "{context} Reply with the label only."
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")
for s in ("train", "dev", "test"):          # the other prompt variant: the manifest's own label-list prompt
    (d / f"{s}p1_ft.json").write_text((d / f"{s}.json").read_text())
print({s: sum(1 for _ in open(d / f"{s}.json")) for s in ("train", "dev", "test")})

## 2. Baseline

In [ ]:
TASK = f"--task cls --max-tokens 16 --labels {d}/labels.txt"
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt-template {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = 100 * evaluate(CKPT, d / f"{s}.json", f"genre_base_{p}_{s}", TASK, extra)["score"]
    print(f"{p}  dev {base[p, 'dev']:5.2f}%   test {base[p, 'test']:5.2f}%")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}%")

## 3. Why not fine-tune the weights?

The standard recipe from the other tutorials, and two variations of it, do not beat the base model on this data:

| recipe | behavior |
|---|---|
| encoder + LoRA, full clips, 300 steps | small dev gain that does not transfer: below the base model on test |
| 10-s crops (4× examples), encoder LR ×3 | collapses to a single genre for every clip |
| 10-s crops, encoder frozen, LoRA only | worse than the base model |

With 443 artist-disjoint training clips, fine-tuning overwrites what the model already knows about genres faster than
it learns anything that transfers, and a few-point dev gain on 197 clips is within noise.

> When the base model is already strong and the data is tiny, weight fine-tuning can erase more than it adds. Change
> the approach rather than tuning the knobs.

## 4. The recipe: keep the model frozen and probe its label probabilities

The *generated* answer is one word, but the model holds a whole **distribution** over genres. `vllm_label_scores.py`
reads it out exactly: for each clip it appends each genre name (plus `<|im_end|>`) to the prompt, and vLLM returns
the log-probability of those tokens (`prompt_logprobs`). That gives ten numbers per clip, for the full clip and for
three 10-second windows, under both prompts.

In [ ]:
G = WORK / "genre_scores"
G.mkdir(exist_ok=True)
for s in ("train", "dev", "test"):
    for p, suffix in (("p2", ""), ("p1", "p1")):
        out = G / f"{p}_{s}.jsonl"
        sh(f"{VLLM_PYTHON} {FT}/vllm_label_scores.py --model {CKPT} --manifest {d}/{s}{suffix}_ft.json "
           f"--labels {d}/labels.txt --windows 10 --out {out} --reuse", log=f"genre_scores_{p}_{s}.log", retries=RETRIES)
row = json.loads(open(G / "p2_dev.jsonl").readline())
print(row["id"], row["window"], {k: round(v, 2) for k, v in sorted(row["scores"].items(), key=lambda kv: -kv[1])[:4]})

Sanity check: the argmax of the full-clip scores lands close to the zero-shot dev accuracy (75.6% vs 76.1% for the
generated answer).

The base model never trained on the 443 training clips, so they're fresh data for fitting a small **multinomial
logistic regression** on top of its scores: 60 features (2 prompts × {full clip, mean over windows, max over windows}
× 10 genres) and about 600 parameters. `label_probe.py` chooses the regularization by **5-fold cross-validation on
train**, checks dev, refits on train + dev, and scores test once.

In [ ]:
sh(f"{VLLM_PYTHON} {FT}/label_probe.py --pool-max --fit-on-dev-too "
   f"--train {G}/p2_train.jsonl {G}/p1_train.jsonl --dev {G}/p2_dev.jsonl {G}/p1_dev.jsonl "
   f"--test {G}/p2_test.jsonl {G}/p1_test.jsonl --out {EVAL}/genre_probe_TEST.jsonl", log="genre_probe.log")
print(open(WORK / "logs" / "genre_probe.log").read())
rows = [json.loads(l) for l in open(EVAL / "genre_probe_TEST.jsonl")]
PROBE = 100 * sum(r["pred_text"] == r["text"] for r in rows) / len(rows)
print(f"test accuracy: base {BASELINE:.2f}% → probe {PROBE:.2f}% ({(PROBE - BASELINE) / BASELINE:+.1%})")
result("09 baseline test accuracy", BASELINE, 78.97, 1.0)  # ±2-3 of 290 clips run to run
result("09 probe test accuracy", PROBE, 86.21, 1.0)

Expected results. Dev accuracy (regularization chosen by cross-validation on train, l2=100):

| probe features | dev |
|---|---:|
| raw zero-shot (argmax of p2 full-clip scores) | 75.63% |
| **p2 + p1: full + mean + max** | **83.25%** |

| | test accuracy |
|---|---:|
| base, prompt chosen on dev (p2) | 78.97% |
| **frozen model + probe** | **86.21%** |

**+9.2% relative.** The probe lands above the ~79-80% reported for this split by dedicated music models, and the
model's weights never change.

## Takeaways

1. **Check the target against the literature first.** +10% over a strong prompted baseline can exceed the state of
   the art.
2. **Fine-tuning can destroy knowledge.** On tiny data with a strong prior, check whether training helps at all before
   tuning it.
3. **A frozen LLM's label probabilities make excellent features.** A 600-parameter probe, fitted on data the model
   hadn't seen, took the frozen model from 79.0% to 86.2%.